# Run Inside an Allocation

## What you'll learn

- Use an existing SLURM allocation with `SlurmIntraRunner`
- Bound concurrent `srun` worker tasks using Artisan's batching settings
- Verify the same CSV and metric results as the queue-based workflow

**Prerequisites:** Follow [Start the allocation tutorial](index.md#start-the-allocation-tutorial).
Start Jupyter and its Python kernel from the shell that owns the allocation.
Opening `salloc` in another terminal does not move an existing kernel into
that allocation. The environment and tutorial directories must be shared
with its compute nodes.
**Estimated time:** 10 minutes after allocation startup.
**GPU required:** No.

Reserve at least two CPUs and 4 GiB of memory. Partition and account are
chosen by `salloc` or `sbatch`, before the kernel starts; they are not
`SlurmIntraRunner` options. Exit the allocation shell after closing Jupyter.

## Check the kernel's allocation

This notebook checks the kernel environment before clearing any previous
tutorial results. The installed built-in operations are importable in
the worker processes created on the allocated node.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import artisan
import polars as pl
from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.utils import tutorial_setup
from artisan.visualization import (
    inspect_data,
    inspect_metrics,
    inspect_pipeline,
    inspect_step,
)

import artisan_submitit
from artisan_submitit import SlurmIntraRunner

print(f"Artisan {artisan.__version__}: {Path(artisan.__file__).resolve()}")
print(
    f"artisan-submitit {artisan_submitit.__version__}: {Path(artisan_submitit.__file__).resolve()}"
)

allocation_id = os.environ.get("SLURM_JOB_ID")
if not allocation_id:
    message = (
        "Start this notebook kernel inside a SLURM allocation; SLURM_JOB_ID is absent"
    )
    raise RuntimeError(message)
print(f"Existing allocation: {allocation_id}")

## Choose a shared workspace

`ARTISAN_TUTORIAL_ROOT` must name a directory visible at the same absolute path
on the manager and compute nodes. The default, `tutorial-runs` under the
notebook's working directory, is suitable only when that location is shared.
Data, staging, working files, and logs stay beneath this tutorial's directory.

The next cell calls `tutorial_setup(..., clean=True)`, which deletes this
notebook's previous `runs/inside-an-allocation` directory before creating a fresh store.
Finish or cancel previous jobs before rerunning setup. Save results you want
to keep elsewhere, then restart the kernel and run all cells in order.

In [ ]:
shared_root = Path(
    os.environ.get("ARTISAN_TUTORIAL_ROOT", str(Path.cwd() / "tutorial-runs"))
).expanduser()
if not shared_root.is_absolute():
    message = "ARTISAN_TUTORIAL_ROOT must be an absolute shared path"
    raise ValueError(message)
shared_root = shared_root.resolve()

env = tutorial_setup("inside-an-allocation", base_dir=shared_root, clean=True)
runner = SlurmIntraRunner(log_folder=str(Path(env.runs_dir) / "logs"))


resources = {"cpus": 1, "memory_gb": 1, "time_limit": "00:05:00"}
print(f"Tutorial results: {env.runs_dir}")

## Run the workflow with `srun`

`SlurmIntraRunner` starts synchronous steps inside the current allocation.
Each task requests one CPU and 1 GiB; `max_workers=2` is the concurrency
ceiling for a step. The five-minute task limit must fit within the
allocation's remaining time.

All three operations use the pipeline default runner. Four seeded CSVs
become four transformed CSVs and four metric artifacts. The data wiring
is unchanged from the [queue tutorial](01-first-slurm-pipeline.ipynb).
The `finally` block closes the pipeline if an operation raises.

In [ ]:
pipeline = PipelineManager.create(
    name="inside_allocation",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
)
output = pipeline.output

try:
    generated = pipeline.run(
        DataGenerator,
        name="generate",
        params={"count": 4, "rows_per_file": 10, "seed": 42},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
    transformed = pipeline.run(
        DataTransformer,
        name="transform",
        inputs={"dataset": output("generate", "datasets")},
        params={"scale_factor": 0.5, "noise_amplitude": 0, "variants": 1, "seed": 100},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
    measured = pipeline.run(
        MetricCalculator,
        name="metrics",
        inputs={"dataset": output("transform", "dataset")},
        runner_resources=resources,
        batch_strategy={"max_workers": 2},
    )
finally:
    summary = pipeline.finalize()

assert summary["overall_success"], summary
assert summary["total_steps"] == 3
run_id = pipeline.config.pipeline_run_id
inspect_pipeline(env.delta_root, pipeline_run_id=run_id)

## Check the data

Each step should expose four accepted artifacts in this run. The source
and transformed datasets contain forty rows each, and every numeric value
should be halved with the operation's four-decimal rounding.

The generator still reports one successful generative execution, rather
than four successful inputs. Inspect artifact counts separately. The `_source` column contains artifact
names without the `.csv` extension.

In [ ]:
for step in (generated, transformed, measured):
    artifacts = inspect_step(env.delta_root, step.step_number, pipeline_run_id=run_id)
    assert artifacts.height == 4, (step.step_name, artifacts)
assert generated.succeeded_count == 1

source_data = inspect_data(
    env.delta_root, step_number=generated.step_number, pipeline_run_id=run_id
)
transformed_data = inspect_data(
    env.delta_root, step_number=transformed.step_number, pipeline_run_id=run_id
)
assert source_data.height == transformed_data.height == 40
assert source_data["_source"].n_unique() == transformed_data["_source"].n_unique() == 4

for name in sorted(source_data["_source"].unique()):
    source = source_data.filter(pl.col("_source") == name).sort("id")
    actual = transformed_data.filter(pl.col("_source") == f"{name}_0").sort("id")
    assert source.height == actual.height == 10
    assert source["id"].to_list() == actual["id"].to_list()
    for column in ("x", "y", "z", "score"):
        expected = [round(value * 0.5, 4) for value in source[column]]
        assert all(
            abs(found - wanted) < 1e-8
            for found, wanted in zip(actual[column], expected, strict=True)
        ), (name, column)

transformed_data.head()

## Read the metrics

The metric table should contain four datasets with ten rows each, matching
the queue-based workflow.

In [ ]:
metrics = inspect_metrics(
    env.delta_root, step_number=measured.step_number, pipeline_run_id=run_id
)
assert metrics.height == 4
assert metrics["summary.row_count"].to_list() == [10, 10, 10, 10]
metrics

## Next steps

- [Batching and managed commands](03-batching-and-commands.ipynb) shows how
  input grouping differs from worker packing.
- [Intra-allocation execution](../srun.md) covers supported arguments,
  task limits, and cancellation within the allocation.
- Return to [tutorial setup](index.md) before running a queue-based
  notebook from a login or workflow node.

Close Jupyter and leave the allocation shell when you finish. Wait for or
cancel previous tutorial jobs before rerunning the setup cell.